# 06. Text Regression, Continuous Scoring & Readability

> **Kategori**: NLP Competition Starter Template
> **Deskripsi**: Pipeline end-to-end untuk kasus pemodelan teks bernilai kontinu (essay scoring, sentiment intensity 0-10, readability index, engagement prediction) menggunakan TF-IDF Ridge, GBDT, dan IndoBERT Regression Head.


### 1. Import Libraries & Setup Environment


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import lightgbm as lgb

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from tabular_pipeline.text_engine import UniversalTextCleaner, TextFeatureExtractor
print("Framework ready. ROOT_DIR:", ROOT_DIR)


### 2. Load Dataset & Auto-Detect Columns


In [ ]:
# Ganti path dataset sesuai dengan data kompetisi
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"

# Contoh dummy dataframe jika dijalankan mandiri
if not os.path.exists(TRAIN_PATH):
    df_train = pd.DataFrame({
        "id": range(1, 101),
        "text": ["Esai tentang pengaruh AI dalam pendidikan dan pembelajaran modern " * 3] * 100,
        "score": np.random.uniform(50.0, 95.0, 100)
    })
else:
    df_train = pd.read_csv(TRAIN_PATH)

TEXT_COL = "text" if "text" in df_train.columns else df_train.select_dtypes(include="object").columns[0]
TARGET_COL = "score" if "score" in df_train.columns else df_train.select_dtypes(include=np.number).columns[-1]

print(f"Text Column: '{TEXT_COL}' | Target Column: '{TARGET_COL}'")
print(df_train[[TEXT_COL, TARGET_COL]].head())


### 3. Text Cleaning & Preprocessing


In [ ]:
cleaner = UniversalTextCleaner()
df_train["clean_text"] = df_train[TEXT_COL].apply(lambda s: cleaner.clean_minimal(str(s)))
print("Text sample cleaned:")
print(df_train[["clean_text", TARGET_COL]].head())


### 4. Stylometrics & Text Morphological Features


In [ ]:
# Ekstraksi fitur gaya bahasa: panjang karakter, jumlah kata, rasio huruf kapital, keragaman leksikal
style_df = TextFeatureExtractor.extract_stylometrics(df_train, "clean_text")
print("Stylometric features shape:", style_df.shape)
print(style_df.head())


### 5. Baseline Model: TF-IDF + Ridge Regression (5-Fold CV)


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=2026)
vec = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_tfidf = vec.fit_transform(df_train["clean_text"])
y = df_train[TARGET_COL].values

oof_ridge = np.zeros(len(df_train))
rmses, maes, r2s = [], [], []

for fold, (trn_idx, val_idx) in enumerate(kf.split(X_tfidf, y), 1):
    X_tr, y_tr = X_tfidf[trn_idx], y[trn_idx]
    X_va, y_va = X_tfidf[val_idx], y[val_idx]
    
    model = Ridge(alpha=1.0)
    model.fit(X_tr, y_tr)
    pred_va = model.predict(X_va)
    oof_ridge[val_idx] = pred_va
    
    rmses.append(np.sqrt(mean_squared_error(y_va, pred_va)))
    maes.append(mean_absolute_error(y_va, pred_va))
    r2s.append(r2_score(y_va, pred_va))

print(f"Ridge CV 5-Fold -> RMSE: {np.mean(rmses):.4f} | MAE: {np.mean(maes):.4f} | R2: {np.mean(r2s):.4f}")


### 6. LightGBM on Text Embeddings & Stylometrics


In [ ]:
# Gabungkan TF-IDF SVD features dengan stylometric features
lsa_df = TextFeatureExtractor.extract_lsa(df_train, "clean_text", n_components=30)
X_tab = pd.concat([style_df, lsa_df], axis=1)

oof_lgb = np.zeros(len(df_train))
for fold, (trn_idx, val_idx) in enumerate(kf.split(X_tab, y), 1):
    X_tr, y_tr = X_tab.iloc[trn_idx], y[trn_idx]
    X_va, y_va = X_tab.iloc[val_idx], y[val_idx]
    
    reg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, random_state=2026, verbose=-1)
    reg.fit(X_tr, y_tr)
    oof_lgb[val_idx] = reg.predict(X_va)

lgb_rmse = np.sqrt(mean_squared_error(y, oof_lgb))
print(f"LightGBM Features -> RMSE: {lgb_rmse:.4f}")


### 7. Ensembling Ridge + LightGBM Regressors


In [ ]:
# Linear Blending bobot kontinu
best_w = 0.5
best_rmse = 999999
for w in np.linspace(0, 1, 101):
    blend = w * oof_ridge + (1 - w) * oof_lgb
    score = np.sqrt(mean_squared_error(y, blend))
    if score < best_rmse:
        best_rmse = score
        best_w = w

print(f"Optimal Ensemble Weight: {best_w:.2f} * Ridge + {1-best_w:.2f} * LightGBM")
print(f"Ensemble OOF RMSE: {best_rmse:.4f}")
